In [1]:
import numpy as np
import pandas as pd
import matplotlib
import sklearn

In [2]:
df = pd.read_csv("creditcard.csv")
df.head()



,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [3]:
df.shape

(284807, 31)

In [4]:
# We Check the class Imbalance to understand how rare the fraud is

df["Class"].value_counts()

Class
0    284315
1       492
Name: count, dtype: int64

In [5]:
# X and Y
y = df["Class"]
X = df.drop("Class", axis=1)

print("X Shape: ", X.shape)
print ("Y Shape: ", y.shape)

X Shape:  (284807, 30)
Y Shape:  (284807,)


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
X, y, test_size=0.20, random_state= 42,
stratify=y # perserves fraud ratios
)

print("Y train value counts: ", Y_train.value_counts())

print("Y test value counts: ", Y_test.value_counts())

# We can see that stratisfy worked and we didn't lose our fraud rate. 

Y train value counts:  Class
0    227451
1       394
Name: count, dtype: int64
Y test value counts:  Class
0    56864
1       98
Name: count, dtype: int64


In [25]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

model = RandomForestClassifier(n_estimators=50, class_weight="balanced", random_state=42)
model.fit(X_train, Y_train)
Y_pred = model.predict(X_test)

f1 = f1_score(Y_test, Y_pred)
print("F1 Score: ", f1)

F1 Score:  0.8323699421965318


0.8804347826086957

In [34]:
# Install XGBoost if you haven't already
# pip install xgboost

from xgboost import XGBClassifier
from sklearn.metrics import f1_score, confusion_matrix

# Calculate the imbalance ratio
# Normal / Fraud
ratio = (Y_train == 0).sum() / (Y_train == 1).sum()

# Initialize XGBoost
model = XGBClassifier(
    n_estimators=200,          # number of trees
    max_depth=6,               # tree depth
    learning_rate=0.1,         # step size shrinkage
    scale_pos_weight=ratio,    # handle imbalance
    subsample=0.8,             # row sampling
    colsample_bytree=0.8,      # feature sampling
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

# Train the model
model.fit(X_train, Y_train)

# Predict probabilities
Y_probs = model.predict_proba(X_test)[:,1]

# Adjust threshold for better recall
threshold = 0.3  # lower than 0.5 to catch more fraud
Y_pred = (Y_probs > threshold).astype(int)

# Evaluate
f1 = f1_score(Y_test, Y_pred)
cm = confusion_matrix(Y_test, Y_pred)

print("F1 Score:", f1)
print("Confusion Matrix:\n", cm)

/Users/jaredefrem/CodingProjects/ML_skitlearn_practice/CreditCard_fraud_detector/.venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [15:46:57] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


F1 Score: 0.841025641025641
Confusion Matrix:
 [[56849    15]
 [   16    82]]
